![DB Academy](https://files.training.databricks.com/binder/prod_main/get-started-with-databricks-for-data-warehousing-en_us-3.0.3/images/20261012T131742Z/Get Started with Databricks for Data Warehousing/Includes/images/db-academy.png)

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">
<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lab</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Create a Metric View for Store Performance</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Build a second metric view by assembling YAML blocks in the Catalog Explorer UI, then query it.</div>
</div>
</div>


## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

This notebook runs on Serverless compute. Before running any cells, confirm Serverless is selected in the compute dropdown at the top-right of this notebook. This course was developed and tested on <strong>Serverless environment version 5</strong>.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema so your queries run against the correct context.


In [0]:
%run ./Includes/Classroom-Setup-11

**Context.** In Lesson 10 you read a metric view that the course created for you (`vw_product_metrics`). In this lab you'll create your own, **`vw_store_metrics`**, focused on store-level revenue. It has the same shape as `vw_product_metrics` but joins `dim_store` instead of `dim_product`.

You'll create it the modern way: describe what you want to **Genie Code** and let it generate the metric-view definition. A manual block-by-block reference is provided too, in case you want to see the YAML or Genie isn't available.

The lab has four steps: create the metric view (with Genie Code), confirm it exists, and run two queries against it.


### Step 1: Create the metric view with Genie Code

You'll create `vw_store_metrics` by describing it to **Genie Code** and running the SQL it generates. Open Genie Code (Cmd/Ctrl + I) in the cell below, or use the full Genie / Assistant panel, and give it this prompt:

> Create a Unity Catalog metric view named vw_store_metrics in the vintage_audio schema, using CREATE VIEW with METRICS LANGUAGE YAML. The source is the fact_orders table. Join dim_store on store_key. Add dimensions region and state from dim_store. Add measures total_revenue as SUM of total_amount and order_count as COUNT of order_id.

Run what Genie generates. If it succeeds, skip to Step 2. Compare against the reference below, or use it if you'd rather not use Genie.

In [0]:
%sql
-- Use Genie Code (Cmd/Ctrl + I) with the prompt above to generate the CREATE VIEW statement, then run it.

<details>
<summary><strong>Here is the full CREATE statement</strong></summary>
<div style="position:relative; margin-top:8px;">
<button onclick="var t=document.getElementById('cbMetrics').innerText;var ok=function(b){b.textContent='Copied';setTimeout(function(){b.textContent='Copy';},1500);};var b=this;try{navigator.clipboard.writeText(t).then(function(){ok(b);},function(){fb();});}catch(e){fb();}function fb(){var ta=document.createElement('textarea');ta.value=t;ta.style.position='fixed';ta.style.opacity='0';document.body.appendChild(ta);ta.focus();ta.select();try{document.execCommand('copy');ok(b);}catch(err){}document.body.removeChild(ta);}" style="position:absolute; top:8px; right:8px; background:#1B5162; color:#fff; border:none; border-radius:5px; padding:6px 12px; font-size:13px; cursor:pointer; z-index:1;">Copy</button>
<pre style="background:#F9F7F4; border-radius:6px; padding:14px; margin:0; font-size:14px; overflow:auto;"><code id="cbMetrics">CREATE OR REPLACE VIEW vw&#95;store&#95;metrics
WITH METRICS LANGUAGE YAML AS $$
version: 1.1
source: fact&#95;orders
joins:
  - name: dim&#95;store
    source: dim&#95;store
    "on": dim&#95;store.store&#95;key = source.store&#95;key
dimensions:
  - name: region
    expr: dim&#95;store.region
    display&#95;name: Region
  - name: state
    expr: dim&#95;store.state
    display&#95;name: State
measures:
  - name: total&#95;revenue
    expr: SUM(source.total&#95;amount)
    display&#95;name: Total Revenue
  - name: order&#95;count
    expr: COUNT(source.order&#95;id)
    display&#95;name: Order Count
$$;</code></pre>
</div>
</details>

**Prefer to build it in the UI instead?** You can also create the metric view in Catalog Explorer by assembling the YAML by hand. Expand the steps below.

<details>
<summary>Manual UI alternative (Catalog Explorer)</summary>
<div style="margin-top:8px;">
<ol>
<li>In the sidebar, open <strong>Catalog</strong> &rarr; your <code>labuser...</code> catalog &rarr; <code>vintage_audio</code> &rarr; <code>fact_orders</code>.</li>
<li>Click <strong>Create &gt; Metric view</strong>. Set the catalog to your <code>labuser...</code> catalog, schema to <code>vintage_audio</code>, name to <code>vw_store_metrics</code>, and click <strong>Create</strong>.</li>
<li>Click the <strong>YAML</strong> toggle, delete any auto-generated sample content, and paste the YAML body from the reference above (everything between <code>$$</code> markers: the <code>version</code> through the <code>measures</code> block).</li>
<li>Click <strong>Save</strong>. If the UI reports a syntax error, check indentation: each <code>- name:</code> aligns within its block, and fields beneath are indented two more spaces.</li>
</ol>
</div>
</details>


### Step 2: Confirm the metric view exists

Run the cell below to confirm your metric view was created in the correct schema.


In [0]:
%sql
SHOW VIEWS LIKE 'vw_store_metrics';

You should see one row listing `vw_store_metrics`. If the result is empty, return to the Catalog Explorer tab and confirm the metric view was saved in the `vintage_audio` schema of your `labuser...` catalog.


### Step 3: Query the metric view

Now query your new metric view using the same three rules from Lesson 10: no `SELECT *`, wrap measures in `MEASURE()`, use `GROUP BY ALL`.

#### Query 1: Revenue and order count by region

Run the cell below. This query asks for total revenue and order count, grouped by region.


In [0]:
%sql
SELECT
  region,
  MEASURE(total_revenue) AS revenue,
  MEASURE(order_count)   AS orders
FROM vw_store_metrics
GROUP BY ALL
ORDER BY revenue DESC;

You should see four rows, one per region (`West`, `Midwest`, `Southeast`, `Northeast`). The **West** region typically leads on revenue since it has the most stores (10), followed by Midwest and Southeast (7 each), then Northeast (6).


#### Query 2: Top 10 states by revenue

Run the cell below. This query asks for the top 10 states by revenue.


In [0]:
%sql
SELECT
  state,
  MEASURE(total_revenue) AS revenue,
  MEASURE(order_count)   AS orders
FROM vw_store_metrics
GROUP BY ALL
ORDER BY revenue DESC
LIMIT 10;

You should see 10 rows. Each row is one US state with the total revenue from all stores in that state and the number of orders.


## Nice work
<div style="border-left: 4px solid #4caf50; background: #e8f5e9; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
<div style="color:#333;">
In this lab you:

- Created a new metric view (<code>vw_store_metrics</code>) using <strong>Genie Code</strong> to generate the definition.
- Confirmed it was registered correctly with <code>SHOW VIEWS</code>.
- Queried it with the <code>MEASURE()</code> function and <code>GROUP BY ALL</code> pattern.

Your schema now has two metric views. <code>vw_product_metrics</code> from Lesson 10 covers product-level metrics (category, brand). <code>vw_store_metrics</code> covers store-level metrics (region, state). Both metric views query the same <code>fact_orders</code> table but expose different angles on it. Different stakeholders can query the one that fits their question, with the same trustworthy measure definitions underneath.

This is the final hands-on lesson of the course. The closing <strong>Summary and Next Steps</strong> notebook recaps what you built and points you toward where to go next.
</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>